# Transformer Internals for Inference

DL6 covers what attention *is*. This note covers what the transformer became once people
had to serve it: KV caching, the attention variants that shrink that cache, FlashAttention,
MoE routing, and sampling. These are the topics an LLM-track interview reaches after you've
shown you know basic attention.

**Prerequisite:** DL6 (attention, multi-head, positional encoding) and mlc4 (attention
implemented from scratch in NumPy).

## Why This Matters

- KV cache: what it stores, why decode needs it, and the memory formula
- The KV-cache reduction ladder: MHA → GQA → MLA, and what each trades away
- FlashAttention: why it's faster without changing the math
- MoE routing at inference and why it complicates serving
- Sampling: greedy, temperature, top-k, top-p, min-p — and what each is for
- Tokenizer effects on cost, because tokens are the billing unit

## 1. The KV Cache

During decode, each new token attends to every previous token. Without caching you'd
recompute K and V for the entire prefix at every step — O(T²) work over a generation.
The KV cache stores K and V for past tokens so each step computes Q, K, V for *one* new
token and attends against the cache.

**This is why decode is memory-bandwidth-bound.** You read a large cache and a large weight
matrix to produce a single token. The arithmetic is trivial; the reads are not.

### The memory formula

For $L$ layers, $H_{kv}$ key/value heads, head dim $d_k$, sequence length $T$, batch $B$:

$$\text{KV bytes} = 2 \times L \times B \times H_{kv} \times T \times d_k \times \text{bytes per element}$$

The leading 2 is for K *and* V. Note it's $H_{kv}$, not $H$ — that distinction is the whole
point of GQA.

In [ ]:
import numpy as np
np.random.seed(0)

def kv_cache_gb(n_layers, n_kv_heads, d_head, seq_len, batch=1, bytes_per=2):
    """bytes_per: 2 = bf16/fp16, 1 = FP8, 0.5 = int4 cache."""
    return 2 * n_layers * batch * n_kv_heads * seq_len * d_head * bytes_per / 1e9

# The KV-cache reduction ladder on a hypothetical 32-layer, 32-Q-head model.
print("KV cache at 32k context, batch 16, 32 layers, d_head=128\n")
print(f"{'variant':<28} {'kv heads':>9} {'bf16 GB':>10} {'FP8 GB':>9} {'vs MHA':>8}")
print("-" * 68)
base = None
for name, kvh in [("MHA (no sharing)", 32), ("GQA 8 groups", 8), ("GQA 4 groups", 4), ("MQA (1 kv head)", 1)]:
    g16 = kv_cache_gb(32, kvh, 128, 32_768, batch=16, bytes_per=2)
    g8  = kv_cache_gb(32, kvh, 128, 32_768, batch=16, bytes_per=1)
    base = base or g16
    print(f"{name:<28} {kvh:>9} {g16:>10.1f} {g8:>9.1f} {base/g16:>7.0f}x")

print()
print("Two independent levers, and they multiply:")
print("  - fewer KV heads (GQA/MQA) shrinks the cache")
print("  - FP8 cache halves it again")
print("Together, 32x GQA-8 + FP8 turns 137 GB into 8.6 GB — the difference")
print("between 'needs a multi-GPU node' and 'fits alongside the weights'.")

## 2. GQA and MLA: Shrinking the Cache

**Multi-Head Attention (MHA)** gives every query head its own K and V. Maximum expressiveness,
maximum cache.

**Multi-Query Attention (MQA)** shares a *single* K/V pair across all query heads. Tiny cache,
but a measurable quality drop and known training instability.

**Grouped-Query Attention (GQA)** is the compromise that won: partition H query heads into
$H_{kv}$ groups, one K/V pair per group. At $H_{kv} = H/4$ you get a 4× cache reduction for
close to no quality loss. This is why nearly every modern open-weight model ships with GQA.

**Multi-head Latent Attention (MLA)** goes further: project K and V down into a shared
low-rank latent vector, cache *that*, and reconstruct per-head K/V on the fly. Trades a little
compute for a much smaller cache — attractive precisely because decode is bandwidth-bound, so
spending FLOPs to save reads is a good deal.

| Variant | Cache size | Quality | Notes |
|---|---|---|---|
| MHA | $H$ KV heads | Best | Original; too expensive at long context |
| GQA | $H_{kv}$ KV heads | ≈ MHA at $H/4$–$H/8$ | The default choice today |
| MQA | 1 KV head | Noticeable drop | Aggressive; niche |
| MLA | Low-rank latent | ≈ MHA reported | More compute, much less cache |

In [ ]:
# GQA in NumPy: the mechanic is that KV heads are REPEATED across query-head groups.
def gqa_attention(Q, K, V, n_q_heads, n_kv_heads, causal=True):
    """
    Q: (T, n_q_heads, d)   K, V: (T, n_kv_heads, d)
    Each group of (n_q_heads // n_kv_heads) query heads shares one KV head.
    """
    assert n_q_heads % n_kv_heads == 0, "q heads must divide evenly into kv heads"
    group = n_q_heads // n_kv_heads
    T, _, d = Q.shape

    # np.repeat expands (T, n_kv, d) -> (T, n_q, d) WITHOUT allocating new cache entries.
    # The cache still only ever stores n_kv heads; this expansion is compute-time only.
    K_exp = np.repeat(K, group, axis=1)
    V_exp = np.repeat(V, group, axis=1)

    scores = np.einsum('qhd,khd->hqk', Q, K_exp) / np.sqrt(d)
    if causal:
        mask = np.triu(np.ones((T, T), dtype=bool), k=1)
        scores = np.where(mask[None, :, :], -np.inf, scores)
    w = np.exp(scores - scores.max(axis=-1, keepdims=True))
    w /= w.sum(axis=-1, keepdims=True)
    return np.einsum('hqk,khd->qhd', w, V_exp)

T, H, H_KV, D = 6, 8, 2, 16
Q = np.random.randn(T, H, D)
K = np.random.randn(T, H_KV, D)
V = np.random.randn(T, H_KV, D)

out = gqa_attention(Q, K, V, n_q_heads=H, n_kv_heads=H_KV)
print(f"Q heads: {H}, KV heads: {H_KV}, group size: {H // H_KV}")
print(f"Output shape: {out.shape}  (same as MHA — GQA is invisible downstream)")
print(f"Cache entries stored per layer: {H_KV} vs {H} for MHA  ->  {H // H_KV}x reduction")

## 3. FlashAttention

Standard attention materializes the full $T \times T$ score matrix in GPU high-bandwidth
memory. At long context that matrix is enormous, and writing then re-reading it is the actual
bottleneck — not the matrix multiplies.

**FlashAttention never materializes it.** It tiles the computation, keeps tiles in fast
on-chip SRAM, and uses the online-softmax trick to accumulate the correct result
block-by-block without ever holding all scores at once.

Two things to be clear about in an interview:

1. **The math is identical.** FlashAttention is not an approximation. Output is exact (up to
   floating-point reassociation). Contrast with sparse or linear attention, which *do* change
   the math.
2. **The win is memory traffic, not FLOPs.** It performs slightly *more* arithmetic
   (recomputation in the backward pass) and is still much faster, because attention was
   IO-bound all along.

Memory drops from O(T²) to O(T), which is what made long context practical at all.

In [ ]:
# Online softmax: the numerical trick that lets FlashAttention process scores in blocks.
# Standard softmax needs the global max and the global sum, so it needs all scores at once.
# Online softmax carries a running (max, sum) and RESCALES as new blocks arrive.

def softmax_standard(x):
    e = np.exp(x - x.max())
    return e / e.sum()

def softmax_online(x, block_size=4):
    """Streams x in blocks, never holding a normalized vector over the full length."""
    m = -np.inf      # running max
    s = 0.0          # running sum of exp(x - m)
    acc = []
    for i in range(0, len(x), block_size):
        blk = x[i:i + block_size]
        m_new = max(m, blk.max())
        # Rescale the running sum to the new max, then add this block's contribution.
        s = s * np.exp(m - m_new) + np.exp(blk - m_new).sum()
        acc.append((blk, m_new))
        m = m_new
    return np.concatenate([np.exp(blk - m) for blk, _ in acc]) / s

x = np.random.randn(16) * 5
a, b = softmax_standard(x), softmax_online(x, block_size=4)
print(f"max abs difference: {np.abs(a - b).max():.2e}")
print(f"both sum to 1: {a.sum():.6f}, {b.sum():.6f}")
print()
print("Identical results, but the online version never held all 16 scores normalized")
print("at once. Scale that to T=128k and the O(T^2) score matrix never exists.")

## 4. Mixture of Experts at Inference

An MoE layer replaces the dense FFN with $N$ expert FFNs plus a router. Each token is routed
to the top-$k$ experts (commonly $k=1$ or $2$), and only those run.

**What this buys:** total parameters — and therefore capacity — grow while FLOPs per token
stay roughly flat.

**What this costs, and what interviews probe:**

- **Memory is not saved.** Every expert must be resident. See llm0 for the arithmetic.
- **Load imbalance.** Routing is data-dependent; a hot expert becomes the straggler for the
  whole batch. Training uses auxiliary load-balancing losses; serving uses capacity factors
  that *drop* overflow tokens.
- **Expert parallelism.** Experts get sharded across GPUs, so every layer now involves an
  all-to-all communication step. On slow interconnect this dominates.
- **Latency variance.** Two identical-length requests can take different times depending on
  which experts they hit — awkward for a p99 SLA.

In [ ]:
# Top-k routing and the load-imbalance problem.
def route(tokens_hidden, router_W, top_k=2, capacity_factor=1.25):
    T, N = tokens_hidden.shape[0], router_W.shape[1]
    logits = tokens_hidden @ router_W
    e = np.exp(logits - logits.max(axis=-1, keepdims=True))
    probs = e / e.sum(axis=-1, keepdims=True)

    topk = np.argsort(-probs, axis=-1)[:, :top_k]
    capacity = int(capacity_factor * T * top_k / N)   # per-expert token budget

    load = np.zeros(N, dtype=int)
    dropped = 0
    for t in range(T):
        for ex in topk[t]:
            if load[ex] < capacity:
                load[ex] += 1
            else:
                dropped += 1               # overflow tokens skip this expert entirely
    return load, capacity, dropped

T, D, N = 512, 64, 8
h = np.random.randn(T, D)

print(f"{'routing':<22} {'expert loads':<34} {'cap':>5} {'dropped':>8}")
print("-" * 74)
# Balanced router
W_bal = np.random.randn(D, N) * 0.3
load, cap, drop = route(h, W_bal, top_k=2)
print(f"{'balanced':<22} {str(load):<34} {cap:>5} {drop:>8}")

# Skewed router: one expert is far more attractive
W_skew = W_bal.copy()
W_skew[:, 0] += 2.0
load, cap, drop = route(h, W_skew, top_k=2)
print(f"{'skewed (expert 0 hot)':<22} {str(load):<34} {cap:>5} {drop:>8}")

print()
print("Dropped tokens silently skip that expert's contribution — a quality hit that")
print("does not show up as an error anywhere. Load-balance loss during training and")
print("capacity-factor tuning at serving both exist to keep this number near zero.")

## 5. Sampling

| Strategy | What it does | Use for |
|---|---|---|
| **Greedy** | Always take argmax | Deterministic extraction, classification |
| **Temperature** | Divide logits by $\tau$ before softmax | Global diversity dial |
| **Top-k** | Keep the k highest-probability tokens | Blunt tail cut; k is distribution-blind |
| **Top-p (nucleus)** | Keep the smallest set with cumulative prob ≥ p | Adaptive — nucleus shrinks when the model is confident |
| **Min-p** | Keep tokens with prob ≥ p × max prob | Scales the cut to model confidence; robust at high temperature |

**Temperature 0 is greedy.** Note that "deterministic" is still not guaranteed in practice —
batching, kernel non-determinism, and MoE routing under varying batch composition can all
shift results between runs. If you need reproducibility, pin it and verify rather than assume.

In [ ]:
def softmax(x):
    e = np.exp(x - x.max())
    return e / e.sum()

def top_p_mask(probs, p):
    order = np.argsort(-probs)
    cum = np.cumsum(probs[order])
    cut = np.searchsorted(cum, p) + 1
    keep = np.zeros_like(probs, dtype=bool)
    keep[order[:cut]] = True
    return keep

def min_p_mask(probs, p):
    return probs >= p * probs.max()

# Compare how each strategy behaves on a CONFIDENT vs an UNCERTAIN distribution.
confident = softmax(np.array([9.0, 3.0, 2.5, 2.0, 1.5, 1.0, 0.5, 0.2]))
uncertain = softmax(np.array([2.2, 2.1, 2.0, 1.9, 1.85, 1.8, 1.75, 1.7]))

print(f"{'distribution':<14} {'strategy':<16} {'tokens kept':>12}")
print("-" * 46)
for label, pr in [("confident", confident), ("uncertain", uncertain)]:
    print(f"{label:<14} {'top-k (k=4)':<16} {4:>12}")
    print(f"{'':<14} {'top-p (p=0.9)':<16} {top_p_mask(pr, 0.9).sum():>12}")
    print(f"{'':<14} {'min-p (p=0.1)':<16} {min_p_mask(pr, 0.1).sum():>12}")
    print()

print("top-k keeps 4 either way — it cannot tell the two apart.")
print("top-p and min-p both narrow when the model is confident and widen when it isn't.")
print("That adaptivity is the entire argument for preferring them.")

## 6. Tokenization Is a Cost Model

Tokens are the billing unit, the context-window unit, and the latency unit. A few
consequences worth carrying into a design discussion:

- English prose runs roughly 4 characters per token. **Code, JSON, and non-Latin scripts run
  far worse** — sometimes 1–2 characters per token. A multilingual product can pay several
  times more per "same" message.
- Whitespace and formatting are billable. Pretty-printed JSON in a prompt costs real money at
  volume.
- Numbers tokenize badly and inconsistently, which is part of why models are shaky at
  arithmetic.
- **Never estimate cost from character counts.** Tokenize a representative sample.

## Common Interview Questions

**Q: Walk me through why decode is memory-bandwidth-bound.**
To emit one token you read the full weight matrices and the whole KV cache from HBM, then do
roughly `2 × params` FLOPs. Modern accelerators have far more FLOP/s than bytes/s, so the read
dominates and the arithmetic units idle. Batching is the fix: reading weights once for 64
sequences amortizes the expensive part, which is exactly why continuous batching matters so
much for throughput.

**Q: Why did GQA become the default rather than MQA?**
MQA collapses to a single KV head — maximum savings, but measurable quality degradation and
training instability. GQA keeps a handful of KV groups, which recovers essentially all the
quality while still cutting the cache 4–8×. It's the knee of the curve, and the knee is where
production settles.

**Q: Is FlashAttention an approximation?**
No — output is exact. It reorders the computation to keep tiles in on-chip SRAM and uses
online softmax to avoid materializing the T×T score matrix. It actually does slightly more
arithmetic (recomputation in backward) and is still much faster, because attention was
IO-bound rather than compute-bound. Sparse and linear attention *are* approximations;
FlashAttention is not.

**Q: A 400B-parameter MoE only activates 30B per token. Can I serve it on one 80GB GPU?**
No. Activation sparsity reduces FLOPs, not residency — all 400B parameters must be in memory
because routing is data-dependent and any expert may be needed for the next token. At FP8
that's roughly 400GB of weights before the KV cache, so you need a multi-GPU node with expert
parallelism. This is the most common MoE misconception.

**Q: When would you use min-p over top-p?**
At high temperature. Top-p operates on cumulative mass, so when temperature flattens the
distribution the nucleus can swell to include genuinely bad tokens. Min-p thresholds relative
to the top token's probability, so the cut tracks model confidence rather than accumulated
mass — it stays sane where top-p degrades.

**Q: Your latency SLA is p99 but you're serving MoE. What's the concern?**
Routing is data-dependent, so per-request latency depends on which experts are hit and how
loaded they are. A hot expert becomes a straggler that delays the whole batch, and expert
parallelism adds an all-to-all per layer whose cost varies with routing. You get a wider
latency distribution than a dense model of equivalent active size — which is a p99 problem
even when p50 looks great.

## Key Takeaways
- KV cache turns decode from O(T²) to O(T) recomputation, at the price of O(T) memory
- Cache formula: `2 × L × B × H_kv × T × d_head × bytes` — note H_kv, not H
- Reduction ladder: MHA → GQA (the default) → MQA (aggressive) → MLA (low-rank latent)
- Cache precision multiplies with head reduction: GQA-8 plus FP8 is ~8× on top of ~4×
- FlashAttention is exact, not approximate; it wins on memory traffic and enables long context
- MoE grows capacity at flat FLOPs but saves no memory, and adds load imbalance and latency variance
- Sampling: top-p and min-p adapt to model confidence; top-k cannot
- Tokenize a real sample before quoting any cost estimate — characters lie, especially for code and non-English text